In [1]:
# Load env variables and create client
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model = "google/gemini-2.5-flash-lite"

In [2]:
# Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def chat(messages, system=None, temperature=1.0, tools=None):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages.extend(messages)

    params = {
        "model": model,
        "messages": all_messages,
        "temperature": temperature,
    }
    if tools:
        params["tools"] = tools

    return client.chat.completions.create(**params)

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    return "Reminder set successfully."


# OpenAI-format tool schemas
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_duration_to_datetime",
            "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
            "parameters": {
                "type": "object",
                "properties": {
                    "datetime_str": {
                        "type": "string",
                        "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter."
                    },
                    "duration": {
                        "type": "number",
                        "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0."
                    },
                    "unit": {
                        "type": "string",
                        "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'."
                    },
                    "input_format": {
                        "type": "string",
                        "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'."
                    }
                },
                "required": ["datetime_str"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_reminder",
            "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string",
                        "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'."
                    },
                    "timestamp": {
                        "type": "string",
                        "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp."
                    }
                },
                "required": ["content", "timestamp"]
            }
        }
    }
]

In [4]:
# Agentic tool loop

import json

TOOL_MAP = {
    "add_duration_to_datetime": add_duration_to_datetime,
    "set_reminder": set_reminder,
}


def run_agent(user_input, system=None):
    messages = []
    add_user_message(messages, user_input)

    while True:
        response = chat(messages, system=system, tools=tools)
        choice = response.choices[0]

        # Add the assistant turn to history (preserves tool_calls metadata)
        messages.append(choice.message)

        if choice.finish_reason == "tool_calls":
            # Execute each tool the model requested
            for tool_call in choice.message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                print(f"[tool] calling {name} with {args}")
                func = TOOL_MAP.get(name)
                result = func(**args) if func else f"Unknown tool: {name}"

                # Return the result to the model
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result),
                })
        else:
            # No more tool calls — return the final text response
            return choice.message.content


# Example
response = run_agent(
    "Set a reminder for my dentist appointment on 2026-07-01 at 10am, "
    "and also tell me what date is 30 days from 2026-06-19."
)
print(response)

[tool] calling set_reminder with {'timestamp': '2026-07-01T10:00:00', 'content': 'Dentist appointment'}
----
Setting the following reminder for 2026-07-01T10:00:00:
Dentist appointment
----
[tool] calling add_duration_to_datetime with {'duration': 30, 'datetime_str': '2026-06-19', 'unit': 'days'}
I've set a reminder for your dentist appointment on July 1, 2026, at 10:00 AM. Also, 30 days from June 19, 2026, will be July 19, 2026.
